In [ ]:
# Local packages
from datetime import datetime
import numpy as np

# 3rd party
import imdb
import pandas as pd
import seaborn as sns
from imdb import IMDbError, IMDbDataAccessError
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
dataset = np.load("Dataset/imdb-user-data/Dataset.npy")
titles = np.load("Dataset/imdb-user-data/movie_titles.npy")

### Data preprocessing

In [ ]:
# Number of movies, users,
# Proportion of ratings, dates

In [ ]:
def convert(date_time):
    # d = day of the month
    # B = full month name
    # Y = full year
    format = '%d %B %Y'
    datetime_str = datetime.strptime(date_time, format)

    return datetime_str

In [ ]:
# Instance of cinemagoer class
ia = imdb.Cinemagoer()

In [ ]:
ia.get_movie('120884')  # This is currently throwing an error, but i believe its bc imdb is actually down and not bc I can't code

In [ ]:
data_dict = {
    'user_id': [data.split(',')[0][2:] for data in dataset],
    'movie_id': [data.split(',')[1][2:] for data in dataset],
    'movie_title': titles,
    'rating': [int(data.split(',')[2]) for data in dataset],
    'date': [convert(data.split(',')[3]) for data in dataset]
    }
df = pd.DataFrame(data_dict)

In [ ]:
# clear memory
data_dict={}

In [ ]:
df.head()

In [ ]:
# TODO - Augment data with movie titles

### EDA

In [ ]:
movies = df['movie_id'].value_counts()
movies

In [ ]:
users = df['user_id'].value_counts()
users

In [ ]:
df.rating.value_counts().sort_index().plot(kind="bar", xlabel="Rating", ylabel="Count")

In [ ]:
df["date"].groupby(df["date"].dt.year).count().plot(kind="bar")

In [ ]:
dates = df["date"].value_counts()
dates

### Matrix Factorisation
Using:
- SKLearn NMF (Non-Negative Matrix Factorisation) algorithm [[1](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)]
- Code from [[2](https://github.com/dinesh-git17/movie_recommendation/tree/main)]

In [ ]:
def create_pivot_table(data, min_ratings=100):
    """
    Creates a pivot table (users x movies) with ratings.
    Only movies with at least min_ratings are retained.

    Arguments:
        data (): TBC
        min_ratings (int): TBC
    Returns:
        TBC
    """
    # Count number of ratings per movie
    ratings_count = data.groupby("movie_id")["rating"].count()
    popular_movies = ratings_count[ratings_count >= min_ratings].index
    filtered_data = data[data["movie_id"].isin(popular_movies)]

    # Create pivot table: rows = user_id, columns = movie_id, values = rating
    pivot = filtered_data.pivot_table(index="user_id", columns="movie_id", values="rating")
    return pivot

def NMF_recommendations(movie_id, pivot, n_components=20, top_n=10):
    """
    Generates recommendations using NMF-based matrix factorization
    Fills missing ratings with 0, factorizes the matrix, and computes cosine similarities
    on the movie latent factors.

    Arguments:
        TBC
    Returns:
        recommendations (): top_n recommendations based on similarity to movie_id
    """
    # Fill missing values with 0
    pivot_filled = pivot.fillna(0)

    # Apply NMF to factorize the matrix into user and movie latent factors
    nmf_model = NMF(n_components=n_components, init="random", random_state=42)
    W = nmf_model.fit_transform(pivot_filled)
    H = nmf_model.components_  # shape: (n_components, n_movies)

    # Transpose H to get movie latent factors: shape (n_movies, n_components)
    movie_factors = H.T
    movie_ids = pivot_filled.columns.tolist()

    # Compute cosine similarity between movies using the latent factors
    similarity_matrix = cosine_similarity(movie_factors)
    similarity_df = pd.DataFrame(
        similarity_matrix, index=movie_ids, columns=movie_ids
    )

    if movie_id not in similarity_df.index:
        raise ValueError(f"Movie '{movie_id}' not found in the dataset.")

    # Get the similarity series for the given movie and sort descending
    similar_movies = (
        similarity_df[movie_id]
        .drop(labels=[movie_id])
        .sort_values(ascending=False)
    )
    recommendations = similar_movies.head(top_n)
    return recommendations

In [ ]:
pivot_table = create_pivot_table(df[0:32000])
pivot_table.head()

In [ ]:
recs = NMF_recommendations("0375063", pivot_table)
recs

### Map suggested movie to actual title from IMDB

In [ ]:
# Build slates
# Cosine similarity? for similarity?